# Model 3 — Iranian License Plate Character Classifier

This notebook trains the third model in the ALPR pipeline:

- Input: a single cropped character image
- Output: one of 32 character classes
- Architecture: ConvNeXt Small
- Framework: PyTorch / torchvision
- Device: Apple MPS if available, CPU fallback only with explicit warning
- Dataset:
  - `Datasets/DS_model_3/train/`
  - `Datasets/DS_model_3/valid/`

No data augmentation is applied because the dataset is already externally augmented.

The class mapping is discovered from folder names, validated, printed, and saved into checkpoints.

## Safety and Quality Contract

This notebook is strict by design:

- Hidden files such as `.DS_Store` are ignored.
- Corrupted images are logged and excluded.
- Train and validation sets are never mixed.
- Folder names are treated as labels exactly as stored on disk.
- Unicode Persian labels are preserved.
- Class mappings are printed and saved.
- The best checkpoint is selected by validation macro F1.
- A final standalone inference cell asks for an input path using `input()`.

In [ ]:
from __future__ import annotations

import os
import json
import math
import time
import copy
import random
import shutil
import platform
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import torchvision
from torchvision import transforms
from torchvision.models import convnext_small, ConvNeXt_Small_Weights

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

ImageFile.LOAD_TRUNCATED_IMAGES = False

warnings.filterwarnings("default")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Platform:", platform.platform())

In [ ]:
@dataclass(frozen=True)
class Model3Config:
    # Project paths
    project_root: Path = Path(".").resolve()
    dataset_root: Path = Path("Datasets/DS_model_3")
    train_dir_name: str = "train"
    valid_dir_name: str = "valid"
    saved_model_dir: Path = Path("Saved_models/model_3")


    expected_num_classes: int = 32
    ignore_hidden_files: bool = True
    valid_image_extensions: Tuple[str, ...] = (
        ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
    )

    # Labels
    expected_labels: Tuple[str, ...] = (
        "الف", "ب", "پ", "ت", "ث", "ج", "د", "ز", "س", "ص",
        "ط", "ع", "ق", "ل", "م", "ن", "ﻫ", "و", "ی",
        "D", "S", "♿︎",
        "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
    )

    # Model
    architecture: str = "convnext_small"
    pretrained: bool = True
    image_size: int = 224

    # Training
    seed: int = 20260531
    epochs: int = 40
    batch_size: int = 64
    num_workers: int = 0
    persistent_workers: bool = False
    pin_memory: bool = False
    learning_rate: float = 2e-4
    weight_decay: float = 0.05
    betas: Tuple[float, float] = (0.9, 0.999)
    label_smoothing: float = 0.05
    gradient_clip_norm: float = 1.0

    # Scheduler
    warmup_epochs: int = 3
    min_lr_factor: float = 0.02

    # Early stopping
    early_stopping_patience: int = 6
    best_metric_name: str = "macro_f1"

    # Validation/inference
    top_k: int = 5

    # Device
    prefer_mps: bool = True
    allow_cpu_fallback: bool = True

    # Output files
    best_checkpoint_name: str = "best_model.pt"
    last_checkpoint_name: str = "last_checkpoint.pt"
    config_name: str = "config.json"
    class_to_idx_name: str = "class_to_idx.json"
    idx_to_class_name: str = "idx_to_class.json"
    invalid_samples_name: str = "invalid_samples.csv"
    training_log_name: str = "training_log.csv"
    metrics_name: str = "validation_metrics.json"


    debug: bool = True
    max_visual_samples_per_class: int = 3
    misclassification_gallery_max_items: int = 64


CFG = Model3Config()


PROJECT_ROOT = CFG.project_root
DATASET_ROOT = PROJECT_ROOT / CFG.dataset_root
TRAIN_DIR = DATASET_ROOT / CFG.train_dir_name
VALID_DIR = DATASET_ROOT / CFG.valid_dir_name
SAVED_DIR = PROJECT_ROOT / CFG.saved_model_dir

BEST_CKPT_PATH = SAVED_DIR / CFG.best_checkpoint_name
LAST_CKPT_PATH = SAVED_DIR / CFG.last_checkpoint_name

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("VALID_DIR:", VALID_DIR)
print("SAVED_DIR:", SAVED_DIR)

In [ ]:
def seed_everything(seed: int) -> None:
    """
    Make the training as deterministic as practical.
    Full bitwise determinism is not always guaranteed on MPS,
    but this controls the main randomness sources.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)


    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        print(f"[WARN] Could not enable deterministic algorithms: {exc}")

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(CFG.seed)
print(f"Seed set to: {CFG.seed}")

In [ ]:
def get_training_device(prefer_mps: bool = True, allow_cpu_fallback: bool = True) -> torch.device:
    if prefer_mps and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("[DEVICE] Using Apple MPS.")
        return device

    if prefer_mps and not torch.backends.mps.is_available():
        message = "[DEVICE] Apple MPS is not available."
        if allow_cpu_fallback:
            print(message + " Falling back to CPU. Training will be slower.")
            return torch.device("cpu")
        raise RuntimeError(message + " CPU fallback is disabled.")

    print("[DEVICE] Using CPU by configuration.")
    return torch.device("cpu")


DEVICE = get_training_device(CFG.prefer_mps, CFG.allow_cpu_fallback)

print("Selected device:", DEVICE)
print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

In [ ]:
def require_dir(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{description} does not exist: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{description} is not a directory: {path}")


require_dir(DATASET_ROOT, "Model 3 dataset root")
require_dir(TRAIN_DIR, "Model 3 train directory")
require_dir(VALID_DIR, "Model 3 valid directory")

SAVED_DIR.mkdir(parents=True, exist_ok=True)

(SAVED_DIR / "plots").mkdir(parents=True, exist_ok=True)
(SAVED_DIR / "misclassification_gallery").mkdir(parents=True, exist_ok=True)
(SAVED_DIR / "example_predictions").mkdir(parents=True, exist_ok=True)

print("[OK] Required paths validated.")
print("[OK] Output directories are ready.")

In [ ]:
def is_hidden_path(path: Path) -> bool:
    return any(part.startswith(".") for part in path.parts)


def is_supported_image(path: Path, extensions: Tuple[str, ...]) -> bool:
    return path.suffix.lower() in extensions


def safe_json_dump(obj: Any, path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def safe_json_load(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def pil_verify_image(path: Path) -> Tuple[bool, Optional[str], Optional[Tuple[int, int]]]:
    """
    Strictly validate image readability.

    Returns:
        ok, error_message, size
    """
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img = img.convert("RGB")
            size = img.size

        if size[0] <= 0 or size[1] <= 0:
            return False, f"Non-positive image size: {size}", None

        return True, None, size

    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return False, repr(exc), None
    except Exception as exc:
        return False, f"Unexpected error: {repr(exc)}", None


def print_boxed(title: str) -> None:
    line = "=" * max(80, len(title) + 8)
    print("\n" + line)
    print(title)
    print(line)

In [ ]:
def discover_class_dirs(root: Path, ignore_hidden: bool = True) -> List[Path]:
    class_dirs = []
    for p in root.iterdir():
        if ignore_hidden and p.name.startswith("."):
            continue
        if p.is_dir():
            class_dirs.append(p)

    class_dirs = sorted(class_dirs, key=lambda x: x.name)
    return class_dirs


train_class_dirs = discover_class_dirs(TRAIN_DIR, CFG.ignore_hidden_files)
valid_class_dirs = discover_class_dirs(VALID_DIR, CFG.ignore_hidden_files)

train_labels = [p.name for p in train_class_dirs]
valid_labels = [p.name for p in valid_class_dirs]

print_boxed("Discovered train labels")
print(train_labels)

print_boxed("Discovered valid labels")
print(valid_labels)

expected_labels_set = set(CFG.expected_labels)
train_labels_set = set(train_labels)
valid_labels_set = set(valid_labels)

if len(train_labels) != CFG.expected_num_classes:
    raise ValueError(
        f"Expected {CFG.expected_num_classes} train class folders, found {len(train_labels)}. "
        f"Labels: {train_labels}"
    )

if len(valid_labels) != CFG.expected_num_classes:
    raise ValueError(
        f"Expected {CFG.expected_num_classes} valid class folders, found {len(valid_labels)}. "
        f"Labels: {valid_labels}"
    )

if train_labels_set != valid_labels_set:
    missing_in_valid = sorted(train_labels_set - valid_labels_set)
    missing_in_train = sorted(valid_labels_set - train_labels_set)
    raise ValueError(
        "Train and valid class folders do not match.\n"
        f"Missing in valid: {missing_in_valid}\n"
        f"Missing in train: {missing_in_train}"
    )

if train_labels_set != expected_labels_set:
    missing_expected = sorted(expected_labels_set - train_labels_set)
    unexpected = sorted(train_labels_set - expected_labels_set)
    raise ValueError(
        "Discovered labels do not match expected 32 labels.\n"
        f"Missing expected labels: {missing_expected}\n"
        f"Unexpected labels: {unexpected}"
    )

class_to_idx: Dict[str, int] = {label: idx for idx, label in enumerate(train_labels)}
idx_to_class: Dict[int, str] = {idx: label for label, idx in class_to_idx.items()}

print_boxed("Final class_to_idx mapping")
for label, idx in class_to_idx.items():
    print(f"{idx:02d}: {label}")

safe_json_dump(class_to_idx, SAVED_DIR / CFG.class_to_idx_name)
safe_json_dump({str(k): v for k, v in idx_to_class.items()}, SAVED_DIR / CFG.idx_to_class_name)

print(f"\n[OK] Saved class mapping to: {SAVED_DIR}")

In [ ]:
@dataclass
class ImageRecord:
    path: Path
    label: str
    class_idx: int
    split: str
    width: int
    height: int


@dataclass
class InvalidImageRecord:
    path: str
    split: str
    label: Optional[str]
    reason: str


def build_validated_index(
    split_dir: Path,
    split_name: str,
    class_to_idx: Dict[str, int],
    cfg: Model3Config,
) -> Tuple[List[ImageRecord], List[InvalidImageRecord]]:
    records: List[ImageRecord] = []
    invalids: List[InvalidImageRecord] = []

    for label, class_idx in class_to_idx.items():
        label_dir = split_dir / label

        if not label_dir.exists() or not label_dir.is_dir():
            invalids.append(
                InvalidImageRecord(
                    path=str(label_dir),
                    split=split_name,
                    label=label,
                    reason="Class directory missing",
                )
            )
            continue

        for path in sorted(label_dir.rglob("*"), key=lambda p: str(p)):
            if cfg.ignore_hidden_files and is_hidden_path(path.relative_to(split_dir)):
                continue

            if path.is_dir():
                continue

            if not is_supported_image(path, cfg.valid_image_extensions):
                invalids.append(
                    InvalidImageRecord(
                        path=str(path),
                        split=split_name,
                        label=label,
                        reason=f"Unsupported file extension: {path.suffix}",
                    )
                )
                continue

            ok, err, size = pil_verify_image(path)
            if not ok:
                invalids.append(
                    InvalidImageRecord(
                        path=str(path),
                        split=split_name,
                        label=label,
                        reason=f"Corrupted/unreadable image: {err}",
                    )
                )
                continue

            assert size is not None
            width, height = size
            records.append(
                ImageRecord(
                    path=path,
                    label=label,
                    class_idx=class_idx,
                    split=split_name,
                    width=width,
                    height=height,
                )
            )

    return records, invalids


train_records, train_invalids = build_validated_index(TRAIN_DIR, "train", class_to_idx, CFG)
valid_records, valid_invalids = build_validated_index(VALID_DIR, "valid", class_to_idx, CFG)

invalid_records = train_invalids + valid_invalids

print_boxed("Dataset validation summary")
print(f"Valid train images: {len(train_records):,}")
print(f"Valid valid images: {len(valid_records):,}")
print(f"Invalid/skipped files: {len(invalid_records):,}")

invalid_df = pd.DataFrame([asdict(x) for x in invalid_records])
invalid_report_path = SAVED_DIR / CFG.invalid_samples_name
invalid_df.to_csv(invalid_report_path, index=False, encoding="utf-8-sig")

print(f"Invalid sample report saved to: {invalid_report_path}")

if len(train_records) == 0:
    raise RuntimeError("No valid training images found.")

if len(valid_records) == 0:
    raise RuntimeError("No valid validation images found.")

In [ ]:
def records_to_dataframe(records: List[ImageRecord]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "path": [str(r.path) for r in records],
            "label": [r.label for r in records],
            "class_idx": [r.class_idx for r in records],
            "split": [r.split for r in records],
            "width": [r.width for r in records],
            "height": [r.height for r in records],
        }
    )


train_df = records_to_dataframe(train_records)
valid_df = records_to_dataframe(valid_records)

train_counts = train_df["label"].value_counts().sort_index()
valid_counts = valid_df["label"].value_counts().sort_index()

dist_df = pd.DataFrame({
    "train_count": train_counts,
    "valid_count": valid_counts,
}).fillna(0).astype(int)

display(dist_df)

dist_path = SAVED_DIR / "class_distribution.csv"
dist_df.to_csv(dist_path, encoding="utf-8-sig")
print(f"Class distribution saved to: {dist_path}")

plt.figure(figsize=(16, 6))
x = np.arange(len(dist_df.index))
plt.bar(x - 0.2, dist_df["train_count"], width=0.4, label="train")
plt.bar(x + 0.2, dist_df["valid_count"], width=0.4, label="valid")
plt.xticks(x, dist_df.index, rotation=45, ha="right")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Image Count")
plt.legend()
plt.tight_layout()
plt.savefig(SAVED_DIR / "plots" / "class_distribution.png", dpi=160)
plt.show()

In [ ]:
def serialize_config(cfg: Model3Config) -> Dict[str, Any]:
    data = asdict(cfg)
    for k, v in list(data.items()):
        if isinstance(v, Path):
            data[k] = str(v)
        elif isinstance(v, tuple):
            data[k] = list(v)
    return data


config_payload = serialize_config(CFG)
config_payload["resolved_paths"] = {
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "train_dir": str(TRAIN_DIR),
    "valid_dir": str(VALID_DIR),
    "saved_dir": str(SAVED_DIR),
}
config_payload["class_to_idx"] = class_to_idx
config_payload["idx_to_class"] = {str(k): v for k, v in idx_to_class.items()}

safe_json_dump(config_payload, SAVED_DIR / CFG.config_name)

print(f"[OK] Config saved to: {SAVED_DIR / CFG.config_name}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

valid_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

inference_transform = valid_transform

print("[OK] Transforms created.")
print("No augmentation is applied.")

In [ ]:
class CharacterImageDataset(Dataset):
    def __init__(
        self,
        records: List[ImageRecord],
        transform: Optional[Any] = None,
    ) -> None:
        self.records = records
        self.transform = transform

        if len(self.records) == 0:
            raise ValueError("Dataset received zero records.")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int, str]:
        record = self.records[index]

        try:
            with Image.open(record.path) as img:
                img = img.convert("RGB")
        except Exception as exc:

            raise RuntimeError(f"Failed to load validated image: {record.path}. Error: {exc}") from exc

        if self.transform is not None:
            img = self.transform(img)

        return img, record.class_idx, str(record.path)


train_dataset = CharacterImageDataset(train_records, transform=train_transform)
valid_dataset = CharacterImageDataset(valid_records, transform=valid_transform)

print("Train dataset size:", len(train_dataset))
print("Valid dataset size:", len(valid_dataset))

In [ ]:
def seed_worker(worker_id: int) -> None:
    worker_seed = CFG.seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(CFG.seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=CFG.pin_memory,
    persistent_workers=CFG.persistent_workers if CFG.num_workers > 0 else False,
    drop_last=False,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
    persistent_workers=CFG.persistent_workers if CFG.num_workers > 0 else False,
    drop_last=False,
)

print("[OK] DataLoaders ready.")
print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))

In [ ]:
def denormalize_tensor(img_tensor: torch.Tensor, mean: List[float], std: List[float]) -> torch.Tensor:
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    return img_tensor.cpu() * std_t + mean_t


def visualize_samples(
    dataset: CharacterImageDataset,
    idx_to_class: Dict[int, str],
    n: int = 24,
    seed: int = 123,
) -> None:
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), k=min(n, len(dataset)))

    cols = 6
    rows = math.ceil(len(indices) / cols)

    plt.figure(figsize=(cols * 2.4, rows * 2.8))

    for i, idx in enumerate(indices):
        img_tensor, label_idx, path = dataset[idx]
        img = denormalize_tensor(img_tensor, IMAGENET_MEAN, IMAGENET_STD)
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(f"{idx_to_class[label_idx]}")
        plt.axis("off")

    plt.suptitle("Random Training Samples", fontsize=16)
    plt.tight_layout()
    plt.savefig(SAVED_DIR / "plots" / "random_training_samples.png", dpi=160)
    plt.show()


visualize_samples(train_dataset, idx_to_class, n=24, seed=CFG.seed)

In [ ]:
def build_convnext_small_classifier(
    num_classes: int,
    pretrained: bool = True,
) -> nn.Module:
    if pretrained:
        try:
            weights = ConvNeXt_Small_Weights.DEFAULT
            model = convnext_small(weights=weights)
            print("[MODEL] Loaded ConvNeXt Small with ImageNet pretrained weights.")
        except Exception as exc:
            print(f"[WARN] Failed to load pretrained weights: {exc}")
            print("[MODEL] Falling back to randomly initialized ConvNeXt Small.")
            model = convnext_small(weights=None)
    else:
        model = convnext_small(weights=None)
        print("[MODEL] Loaded ConvNeXt Small without pretrained weights.")

    if not hasattr(model, "classifier"):
        raise RuntimeError("Unexpected ConvNeXt model structure: missing classifier.")

    last_layer = model.classifier[-1]
    if not isinstance(last_layer, nn.Linear):
        raise RuntimeError(f"Unexpected classifier last layer: {type(last_layer)}")

    in_features = last_layer.in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    print(f"[MODEL] Replaced classification head: {in_features} -> {num_classes}")

    return model


model = build_convnext_small_classifier(
    num_classes=CFG.expected_num_classes,
    pretrained=CFG.pretrained,
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

optimizer = AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
    betas=CFG.betas,
)


def lr_lambda(current_epoch: int) -> float:
    """
    Linear warmup followed by cosine decay.
    Returns multiplicative LR factor.
    """
    if current_epoch < CFG.warmup_epochs:
        return float(current_epoch + 1) / float(max(1, CFG.warmup_epochs))

    progress = (current_epoch - CFG.warmup_epochs) / float(max(1, CFG.epochs - CFG.warmup_epochs))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return CFG.min_lr_factor + (1.0 - CFG.min_lr_factor) * cosine


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

print("[OK] Loss, optimizer, and scheduler initialized.")
print("Initial LR:", optimizer.param_groups[0]["lr"])

In [ ]:
@torch.no_grad()
def compute_topk_accuracy(
    logits: torch.Tensor,
    targets: torch.Tensor,
    top_k: int = 5,
) -> float:
    max_k = min(top_k, logits.shape[1])
    _, pred = logits.topk(max_k, dim=1)
    pred = pred.t()
    correct = pred.eq(targets.view(1, -1).expand_as(pred))
    correct_k = correct[:max_k].reshape(-1).float().sum(0)
    return (correct_k / targets.size(0)).item()


def aggregate_classification_metrics(
    y_true: List[int],
    y_pred: List[int],
    idx_to_class: Dict[int, str],
) -> Dict[str, Any]:
    labels = list(range(len(idx_to_class)))
    target_names = [idx_to_class[i] for i in labels]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0,
        output_dict=True,
    )

    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "classification_report": report,
    }


def get_current_lr(optimizer: torch.optim.Optimizer) -> float:
    return float(optimizer.param_groups[0]["lr"])

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epoch: int,
    cfg: Model3Config,
) -> Dict[str, float]:
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0
    running_topk_sum = 0.0

    pbar = tqdm(loader, desc=f"Train Epoch {epoch}", leave=False)

    for images, targets, paths in pbar:
        images = images.to(device, non_blocking=False)
        targets = targets.to(device, non_blocking=False)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, targets)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss detected at epoch {epoch}: {loss.item()}")

        loss.backward()

        if cfg.gradient_clip_norm is not None and cfg.gradient_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.gradient_clip_norm)

        optimizer.step()

        batch_size = targets.size(0)
        preds = logits.argmax(dim=1)

        running_loss += loss.item() * batch_size
        running_correct += (preds == targets).sum().item()
        running_total += batch_size
        running_topk_sum += compute_topk_accuracy(logits.detach(), targets, top_k=cfg.top_k) * batch_size

        pbar.set_postfix({
            "loss": running_loss / max(1, running_total),
            "acc": running_correct / max(1, running_total),
            f"top{cfg.top_k}": running_topk_sum / max(1, running_total),
        })

    return {
        "train_loss": running_loss / max(1, running_total),
        "train_accuracy": running_correct / max(1, running_total),
        f"train_top{cfg.top_k}_accuracy": running_topk_sum / max(1, running_total),
    }


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    epoch: int,
    cfg: Model3Config,
    idx_to_class: Dict[int, str],
) -> Dict[str, Any]:
    model.eval()

    running_loss = 0.0
    running_total = 0
    running_correct = 0
    running_topk_sum = 0.0

    all_targets: List[int] = []
    all_preds: List[int] = []
    all_paths: List[str] = []
    all_confidences: List[float] = []

    pbar = tqdm(loader, desc=f"Valid Epoch {epoch}", leave=False)

    for images, targets, paths in pbar:
        images = images.to(device, non_blocking=False)
        targets = targets.to(device, non_blocking=False)

        logits = model(images)
        loss = criterion(logits, targets)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite validation loss detected at epoch {epoch}: {loss.item()}")

        probs = torch.softmax(logits, dim=1)
        confs, preds = probs.max(dim=1)

        batch_size = targets.size(0)

        running_loss += loss.item() * batch_size
        running_correct += (preds == targets).sum().item()
        running_total += batch_size
        running_topk_sum += compute_topk_accuracy(logits, targets, top_k=cfg.top_k) * batch_size

        all_targets.extend(targets.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_paths.extend(list(paths))
        all_confidences.extend(confs.cpu().numpy().tolist())

        pbar.set_postfix({
            "loss": running_loss / max(1, running_total),
            "acc": running_correct / max(1, running_total),
            f"top{cfg.top_k}": running_topk_sum / max(1, running_total),
        })

    metric_payload = aggregate_classification_metrics(all_targets, all_preds, idx_to_class)

    result = {
        "valid_loss": running_loss / max(1, running_total),
        "valid_accuracy": running_correct / max(1, running_total),
        f"valid_top{cfg.top_k}_accuracy": running_topk_sum / max(1, running_total),
        "macro_f1": metric_payload["macro_f1"],
        "accuracy": metric_payload["accuracy"],
        "classification_report": metric_payload["classification_report"],
        "y_true": all_targets,
        "y_pred": all_preds,
        "paths": all_paths,
        "confidences": all_confidences,
    }

    return result

In [ ]:
def make_checkpoint_payload(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    epoch: int,
    best_metric: float,
    cfg: Model3Config,
    class_to_idx: Dict[str, int],
    idx_to_class: Dict[int, str],
    metrics: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "epoch": epoch,
        "architecture": cfg.architecture,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_metric": best_metric,
        "best_metric_name": cfg.best_metric_name,
        "class_to_idx": class_to_idx,
        "idx_to_class": {str(k): v for k, v in idx_to_class.items()},
        "num_classes": cfg.expected_num_classes,
        "image_size": cfg.image_size,
        "normalization": {
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
        },
        "config": serialize_config(cfg),
        "metrics": metrics,
    }


def save_checkpoint(payload: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)
    print(f"[CHECKPOINT] Saved: {path}")


def strip_large_validation_arrays(metrics: Dict[str, Any]) -> Dict[str, Any]:
    """
    Keep JSON/report files lightweight.
    """
    cleaned = copy.deepcopy(metrics)
    for key in ["y_true", "y_pred", "paths", "confidences"]:
        cleaned.pop(key, None)
    return cleaned

In [ ]:
history: List[Dict[str, Any]] = []

best_metric = -float("inf")
best_epoch = -1
epochs_without_improvement = 0

training_start_time = time.time()

print_boxed("Starting training")
print(f"Device: {DEVICE}")
print(f"Epochs: {CFG.epochs}")
print(f"Batch size: {CFG.batch_size}")
print(f"Best metric: {CFG.best_metric_name}")

for epoch in range(1, CFG.epochs + 1):
    epoch_start = time.time()

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epoch=epoch,
        cfg=CFG,
    )

    valid_metrics = validate_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE,
        epoch=epoch,
        cfg=CFG,
        idx_to_class=idx_to_class,
    )

    scheduler.step()

    epoch_seconds = time.time() - epoch_start
    current_lr = get_current_lr(optimizer)

    if CFG.best_metric_name == "macro_f1":
        current_best_metric = float(valid_metrics["macro_f1"])
    elif CFG.best_metric_name in ("accuracy", "valid_accuracy"):
        current_best_metric = float(valid_metrics["valid_accuracy"])
    else:
        raise ValueError(f"Unsupported best metric: {CFG.best_metric_name}")

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "epoch_seconds": epoch_seconds,
        **train_metrics,
        "valid_loss": valid_metrics["valid_loss"],
        "valid_accuracy": valid_metrics["valid_accuracy"],
        f"valid_top{CFG.top_k}_accuracy": valid_metrics[f"valid_top{CFG.top_k}_accuracy"],
        "macro_f1": valid_metrics["macro_f1"],
    }

    history.append(row)

    print(
        f"Epoch {epoch:03d}/{CFG.epochs} | "
        f"lr={current_lr:.3e} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"train_acc={row['train_accuracy']:.4f} | "
        f"valid_loss={row['valid_loss']:.4f} | "
        f"valid_acc={row['valid_accuracy']:.4f} | "
        f"macro_f1={row['macro_f1']:.4f} | "
        f"time={epoch_seconds:.1f}s"
    )

    last_payload = make_checkpoint_payload(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        best_metric=best_metric,
        cfg=CFG,
        class_to_idx=class_to_idx,
        idx_to_class=idx_to_class,
        metrics=strip_large_validation_arrays(valid_metrics),
    )
    save_checkpoint(last_payload, LAST_CKPT_PATH)

    if current_best_metric > best_metric:
        best_metric = current_best_metric
        best_epoch = epoch
        epochs_without_improvement = 0

        best_payload = make_checkpoint_payload(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            best_metric=best_metric,
            cfg=CFG,
            class_to_idx=class_to_idx,
            idx_to_class=idx_to_class,
            metrics=strip_large_validation_arrays(valid_metrics),
        )
        save_checkpoint(best_payload, BEST_CKPT_PATH)

        print(f"[BEST] New best {CFG.best_metric_name}: {best_metric:.6f} at epoch {best_epoch}")
    else:
        epochs_without_improvement += 1
        print(f"[EARLY STOP] No improvement for {epochs_without_improvement}/{CFG.early_stopping_patience} epochs.")

    history_df = pd.DataFrame(history)
    history_df.to_csv(SAVED_DIR / CFG.training_log_name, index=False, encoding="utf-8-sig")

    metrics_payload = {
        "best_metric_name": CFG.best_metric_name,
        "best_metric": best_metric,
        "best_epoch": best_epoch,
        "last_epoch": epoch,
        "last_metrics": row,
    }
    safe_json_dump(metrics_payload, SAVED_DIR / CFG.metrics_name)

    if epochs_without_improvement >= CFG.early_stopping_patience:
        print("[EARLY STOP] Patience reached. Stopping training.")
        break

total_training_seconds = time.time() - training_start_time
print_boxed("Training finished")
print(f"Best epoch: {best_epoch}")
print(f"Best {CFG.best_metric_name}: {best_metric:.6f}")
print(f"Total training time: {total_training_seconds / 60:.2f} minutes")

In [ ]:
history_df = pd.read_csv(SAVED_DIR / CFG.training_log_name)

display(history_df.tail())

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["valid_loss"], label="valid_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
plt.plot(history_df["epoch"], history_df["valid_accuracy"], label="valid_accuracy")
plt.plot(history_df["epoch"], history_df["macro_f1"], label="macro_f1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Accuracy / Macro F1")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVED_DIR / "plots" / "training_curves.png", dpi=160)
plt.show()

In [ ]:
def load_model3_checkpoint(
    checkpoint_path: Path,
    device: torch.device,
) -> Tuple[nn.Module, Dict[str, int], Dict[int, str], Dict[str, Any]]:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    loaded_class_to_idx = checkpoint["class_to_idx"]
    loaded_idx_to_class = {int(k): v for k, v in checkpoint["idx_to_class"].items()}

    num_classes = int(checkpoint["num_classes"])
    architecture = checkpoint["architecture"]

    if architecture != "convnext_small":
        raise ValueError(f"Unsupported architecture in checkpoint: {architecture}")

    loaded_model = build_convnext_small_classifier(
        num_classes=num_classes,
        pretrained=False,
    )
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model = loaded_model.to(device)
    loaded_model.eval()

    return loaded_model, loaded_class_to_idx, loaded_idx_to_class, checkpoint


best_model, loaded_class_to_idx, loaded_idx_to_class, best_checkpoint = load_model3_checkpoint(
    BEST_CKPT_PATH,
    DEVICE,
)

print("[OK] Best checkpoint loaded.")
print("Checkpoint epoch:", best_checkpoint["epoch"])
print("Best metric:", best_checkpoint["best_metric_name"], best_checkpoint["best_metric"])

In [ ]:
best_valid_metrics = validate_one_epoch(
    model=best_model,
    loader=valid_loader,
    criterion=criterion,
    device=DEVICE,
    epoch=best_checkpoint["epoch"],
    cfg=CFG,
    idx_to_class=loaded_idx_to_class,
)

print_boxed("Best checkpoint validation metrics")
print(f"Valid loss: {best_valid_metrics['valid_loss']:.6f}")
print(f"Valid accuracy: {best_valid_metrics['valid_accuracy']:.6f}")
print(f"Macro F1: {best_valid_metrics['macro_f1']:.6f}")
print(f"Top-{CFG.top_k} accuracy: {best_valid_metrics[f'valid_top{CFG.top_k}_accuracy']:.6f}")

report_dict = best_valid_metrics["classification_report"]
report_df = pd.DataFrame(report_dict).transpose()
display(report_df)

report_path = SAVED_DIR / "classification_report.csv"
report_df.to_csv(report_path, encoding="utf-8-sig")
print(f"Classification report saved to: {report_path}")

In [ ]:
y_true = best_valid_metrics["y_true"]
y_pred = best_valid_metrics["y_pred"]

labels = list(range(len(loaded_idx_to_class)))
display_labels = [loaded_idx_to_class[i] for i in labels]

cm = confusion_matrix(y_true, y_pred, labels=labels)

plt.figure(figsize=(16, 16))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot(
    include_values=False,
    cmap="Blues",
    ax=plt.gca(),
    xticks_rotation=45,
    colorbar=True,
)
plt.title("Confusion Matrix — Best Model")
plt.tight_layout()
plt.savefig(SAVED_DIR / "plots" / "confusion_matrix.png", dpi=180)
plt.show()

cm_df = pd.DataFrame(cm, index=display_labels, columns=display_labels)
cm_df.to_csv(SAVED_DIR / "confusion_matrix.csv", encoding="utf-8-sig")
print(f"Confusion matrix saved to: {SAVED_DIR / 'confusion_matrix.csv'}")

In [ ]:
per_class_rows = []

for class_idx in labels:
    class_name = loaded_idx_to_class[class_idx]
    indices = [i for i, t in enumerate(y_true) if t == class_idx]

    if len(indices) == 0:
        acc = float("nan")
        total = 0
        correct = 0
    else:
        total = len(indices)
        correct = sum(1 for i in indices if y_pred[i] == y_true[i])
        acc = correct / total

    per_class_rows.append({
        "class_idx": class_idx,
        "class_name": class_name,
        "total": total,
        "correct": correct,
        "accuracy": acc,
    })

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)

per_class_path = SAVED_DIR / "per_class_accuracy.csv"
per_class_df.to_csv(per_class_path, index=False, encoding="utf-8-sig")
print(f"Per-class accuracy saved to: {per_class_path}")

plt.figure(figsize=(16, 6))
plt.bar(per_class_df["class_name"], per_class_df["accuracy"])
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1.0)
plt.title("Per-Class Accuracy")
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(SAVED_DIR / "plots" / "per_class_accuracy.png", dpi=160)
plt.show()

In [ ]:
def load_raw_image_for_display(path: str) -> Image.Image:
    with Image.open(path) as img:
        return img.convert("RGB")


misclassified = []
for true_idx, pred_idx, path, conf in zip(
    best_valid_metrics["y_true"],
    best_valid_metrics["y_pred"],
    best_valid_metrics["paths"],
    best_valid_metrics["confidences"],
):
    if true_idx != pred_idx:
        misclassified.append({
            "path": path,
            "true_idx": true_idx,
            "pred_idx": pred_idx,
            "true_label": loaded_idx_to_class[true_idx],
            "pred_label": loaded_idx_to_class[pred_idx],
            "confidence": conf,
        })

mis_df = pd.DataFrame(misclassified)
mis_path = SAVED_DIR / "misclassifications.csv"
mis_df.to_csv(mis_path, index=False, encoding="utf-8-sig")

print(f"Total misclassified: {len(misclassified):,}")
print(f"Misclassification report saved to: {mis_path}")

if len(misclassified) > 0:
    misclassified_sorted = sorted(misclassified, key=lambda x: x["confidence"], reverse=True)
    examples = misclassified_sorted[:CFG.misclassification_gallery_max_items]

    cols = 8
    rows = math.ceil(len(examples) / cols)

    plt.figure(figsize=(cols * 2.2, rows * 2.8))

    for i, item in enumerate(examples):
        img = load_raw_image_for_display(item["path"])
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"T:{item['true_label']}\nP:{item['pred_label']}\n{item['confidence']:.2f}",
            fontsize=9,
        )

    plt.suptitle("High-Confidence Misclassifications", fontsize=16)
    plt.tight_layout()
    plt.savefig(SAVED_DIR / "plots" / "misclassification_gallery.png", dpi=180)
    plt.show()
else:
    print("No misclassifications found. یا مدل خیلی خوب شده، یا دیتاست زیادی مهربان بوده 😄")

In [ ]:
@torch.no_grad()
def predict_character_image(
    image_path: Path,
    model: nn.Module,
    transform: Any,
    idx_to_class: Dict[int, str],
    device: torch.device,
) -> Dict[str, Any]:
    if not image_path.exists():
        raise FileNotFoundError(f"Input image does not exist: {image_path}")

    if image_path.is_dir():
        raise IsADirectoryError(f"Input path is a directory, expected image file: {image_path}")

    if image_path.name.startswith("."):
        raise ValueError(f"Hidden file is not a valid input image: {image_path}")

    ok, err, size = pil_verify_image(image_path)
    if not ok:
        raise ValueError(f"Invalid/corrupted input image: {image_path}. Reason: {err}")

    with Image.open(image_path) as img:
        pil_img = img.convert("RGB")

    tensor = transform(pil_img).unsqueeze(0).to(device)

    model.eval()
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu()

    pred_idx = int(torch.argmax(probs).item())
    pred_label = idx_to_class[pred_idx]
    confidence = float(probs[pred_idx].item())

    prob_rows = []
    for idx in range(len(idx_to_class)):
        prob_rows.append({
            "class_idx": idx,
            "class_label": idx_to_class[idx],
            "probability": float(probs[idx].item()),
        })

    prob_df = pd.DataFrame(prob_rows).sort_values("probability", ascending=False).reset_index(drop=True)

    return {
        "image": pil_img,
        "image_size": size,
        "pred_idx": pred_idx,
        "pred_label": pred_label,
        "confidence": confidence,
        "probabilities": prob_df,
    }

In [ ]:
input_path_str = input("Enter path to a single cropped character image: ").strip()

try:
    input_path = Path(input_path_str).expanduser().resolve()

    inference_model, inference_class_to_idx, inference_idx_to_class, inference_ckpt = load_model3_checkpoint(
        BEST_CKPT_PATH,
        DEVICE,
    )

    result = predict_character_image(
        image_path=input_path,
        model=inference_model,
        transform=inference_transform,
        idx_to_class=inference_idx_to_class,
        device=DEVICE,
    )

    print_boxed("Prediction")
    print(f"Input image: {input_path}")
    print(f"Image size: {result['image_size']}")
    print(f"Predicted class: {result['pred_label']}")
    print(f"Confidence: {result['confidence']:.6f}")

    plt.figure(figsize=(4, 4))
    plt.imshow(result["image"])
    plt.title(f"Predicted: {result['pred_label']} ({result['confidence']:.3f})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print_boxed("Full 32-Class Confidence Table")
    display(result["probabilities"])

except Exception as exc:
    print("[ERROR] Inference failed.")
    print(type(exc).__name__ + ":", exc)